# 04 — Transformer LTV Model (PyTorch + ONNX)

**Phase 3 — Week 3**

**Goal:** Train a multi-head Transformer on purchase sequences to predict LTV at 12m / 24m / 36m horizons. Export to ONNX for < 200ms serving.

**Architecture:**
- Input: purchase sequence (max 50 tokens)
- Token = [product_category, amount_bucket, days_delta, channel]
- Transformer Encoder: 4 layers, 8 heads, FFN dim 256
- CLS token → 64-dim embedding (stored in pgvector)
- 3 prediction heads: LTV_12m / LTV_24m / LTV_36m
- Loss: multi-horizon Huber on log1p(LTV)

**Targets:**
| Metric | Target |
|---|---|
| MAE LTV 12m | < 15% of mean LTV |
| Gini | > 0.65 |
| Top decile lift | > 3.0× |
| ONNX inference | < 200ms |
| ONNX vs PyTorch MAE | < 1e-5 |


In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import polars as pl
import torch
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from rich import print as rprint
import plotly.io as pio
pio.templates.default = 'plotly_white'

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.db.supabase_client import SupabaseClient
from backend.features.rfm import (
    clean_transactions, assign_product_categories,
    assign_amount_buckets, make_calibration_holdout_split, RFMPipeline
)
from backend.features.sequences import SequenceBuilder
from backend.ml.transformer_model import LTVTransformer, build_model, count_parameters
from backend.ml.sequence_dataset import PurchaseSequenceDataset, make_dataloaders
from backend.ml.trainer import LTVTrainer
from backend.ml.transformer_onnx import export_to_onnx, validate_onnx_vs_pytorch, ONNXInferenceEngine
from backend.ml.transformer_evaluator import evaluate_on_holdout, predict_all_customers
from backend.ml.embedding_store import extract_embeddings, store_embeddings
from backend.ml.optuna_tuner import run_optuna_study
from torch.utils.data import DataLoader
from backend.ml.sequence_dataset import collate_fn

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device: cpu
PyTorch: 2.11.0+cpu


In [2]:
# ── Config ────────────────────────────────────────────────────
MAX_SEQ_LEN         = 50
OBSERVATION_MONTHS  = 6
HOLDOUT_MONTHS      = 6
RUN_OPTUNA          = False   # Set True to tune (slow — ~20 trials)
N_OPTUNA_TRIALS     = 10
TRAIN_EPOCHS        = 50
PATIENCE            = 10
SAVE_TO_DB          = True
USE_WANDB           = True
ONNX_PATH           = settings.MODELS_DIR / 'transformer.onnx'
MODELS_DIR          = settings.MODELS_DIR

print(f'ONNX path: {ONNX_PATH}')

ONNX path: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer.onnx


## 1. Load Data & Build Sequences

In [3]:
# Load and clean
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

# Split (observation anchor + diagnostics)
calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned, OBSERVATION_MONTHS, HOLDOUT_MONTHS
)

# RFM labels
rfm_pipe = RFMPipeline(calibration, observation_end_date=obs_end)
rfm_df   = rfm_pipe.compute()
rfm_df   = rfm_pipe.compute_ltv_labels(cleaned, rfm_df, horizon_months=12)
rfm_df   = rfm_pipe.compute_ltv_labels(cleaned, rfm_df, horizon_months=24)
rfm_df   = rfm_pipe.compute_ltv_labels(cleaned, rfm_df, horizon_months=36)

print(f'Calibration: {len(calibration):,} rows | Holdout: {len(holdout):,} rows')
print(f'RFM customers: {len(rfm_df):,}')
print(f'Non-zero 12m LTV: {(rfm_df["actual_ltv_12m"] > 0).sum():,}')



2026-04-26 01:41:29.934 | INFO     | backend.data.load_data:load_uci_csv:75 - Loading UCI dataset from: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\backend\data\raw\OnlineRetail.csv
2026-04-26 01:41:29.935 | INFO     | backend.data.load_data:load_uci_csv:93 - Reading CSV with DuckDB for robust date parsing...
2026-04-26 01:41:35.274 | INFO     | backend.data.load_data:load_uci_csv:130 - Loaded 541909 rows — 8 columns — date range 2010-12-01 08:26:00+00:00 to 2011-12-09 12:50:00+00:00
2026-04-26 01:41:35.275 | INFO     | backend.features.rfm:clean_transactions:54 - Cleaning transactions — 541909 raw rows
2026-04-26 01:41:35.293 | INFO     | backend.features.rfm:clean_transactions:76 - After cleaning: 397884 rows (73.4% kept)
2026-04-26 01:41:35.320 | INFO     | backend.features.rfm:make_calibration_holdout_split:157 - Split → calibration 144541 rows (≤2011-05-30), holdout 225975 rows (2011-05-30 – 2011-11-26)
2026-04-26 01:41:35.320 | INFO     | backend.feat

Calibration: 144,541 rows | Holdout: 225,975 rows
RFM customers: 2,708
Non-zero 12m LTV: 1,925


In [4]:
# Build purchase sequences from calibration window only
builder_cal  = SequenceBuilder(calibration, max_length=MAX_SEQ_LEN, observation_end_date=str(obs_end))
sequences_df = builder_cal.build()

print(f'Calibration sequences: {len(sequences_df):,}')
print(f'Avg sequence length:   {sequences_df["sequence_length"].mean():.1f}')



2026-04-26 01:41:35.435 | INFO     | backend.features.sequences:build:95 - Building purchase sequences for 144541 transactions (max_length=50)
2026-04-26 01:41:35.554 | INFO     | backend.features.sequences:build:180 - Built 2708 sequences — avg length 2.7, max 50


Calibration sequences: 2,708
Avg sequence length:   2.7


In [5]:
# Build datasets -- customer-level train/val/hold split from calibration sequences
from sklearn.model_selection import train_test_split

all_ids = sequences_df['customer_id'].to_list()
label_map = {
    row['customer_id']: float(row['actual_ltv_12m'])
    for row in rfm_df.select(['customer_id', 'actual_ltv_12m']).iter_rows(named=True)
}
all_y = [1 if label_map.get(cid, 0.0) > 0 else 0 for cid in all_ids]

train_ids, temp_ids, y_train, y_temp = train_test_split(
    all_ids, all_y,
    test_size=0.30,
    random_state=42,
    stratify=all_y,
)
val_ids, hold_ids, _, _ = train_test_split(
    temp_ids, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

train_seq = sequences_df.filter(pl.col('customer_id').is_in(train_ids))
val_seq   = sequences_df.filter(pl.col('customer_id').is_in(val_ids))
hold_seq  = sequences_df.filter(pl.col('customer_id').is_in(hold_ids))

train_dataset = PurchaseSequenceDataset(
    train_seq, rfm_df, max_length=MAX_SEQ_LEN,
    ltv_12m_col='actual_ltv_12m', ltv_24m_col='actual_ltv_24m', ltv_36m_col='actual_ltv_36m'
)
val_dataset = PurchaseSequenceDataset(
    val_seq, rfm_df, max_length=MAX_SEQ_LEN,
    ltv_12m_col='actual_ltv_12m', ltv_24m_col='actual_ltv_24m', ltv_36m_col='actual_ltv_36m'
)
hold_dataset = PurchaseSequenceDataset(
    hold_seq, rfm_df, max_length=MAX_SEQ_LEN,
    ltv_12m_col='actual_ltv_12m', ltv_24m_col='actual_ltv_24m', ltv_36m_col='actual_ltv_36m'
)

print(f'Train: {len(train_dataset):,}  Val: {len(val_dataset):,}  Hold: {len(hold_dataset):,}')



Train: 1,895  Val: 406  Hold: 407


## 2. Optuna Hyperparameter Tuning (optional)

In [6]:
if RUN_OPTUNA:
    rprint('[bold blue]Running Optuna tuning...[/bold blue]')
    best_params, study = run_optuna_study(
        train_dataset, val_dataset,
        n_trials=N_OPTUNA_TRIALS, device=DEVICE, quick_epochs=10
    )
    rprint(f'Best params: {best_params}')
else:
    # Default config from plan
    best_params = {
        'n_layers':      4,
        'n_heads':       8,
        'ffn_dim':       256,
        'dropout':       0.1,
        'learning_rate': 1e-3,
        'weight_decay':  1e-4,
        'batch_size':    64,
    }
    rprint(f'Using default config: {best_params}')

Using default config: {'n_layers': 4, 'n_heads': 8, 'ffn_dim': 256, 'dropout': 0.1, 'learning_rate': 0.001, 
'weight_decay': 0.0001, 'batch_size': 64}

## 3. Train the Transformer

In [7]:
config = {
    'model_dim': 96,
    'max_seq_len': MAX_SEQ_LEN,
    'grad_clip': 1.0,
    'huber_delta': 1.0,
    'loss_weights': [1.0, 0.0, 0.0],
    'positive_ltv_weight': 4.0,
    'train_pos_oversample': 4.0,
    **best_params,
}

model = build_model(config)
rprint(f'Model parameters: {count_parameters(model):,}')

from torch.utils.data import WeightedRandomSampler

is_pos = (train_dataset.ltv_12m > 0).numpy().astype(np.float32)
sample_weights = np.where(is_pos > 0, config['train_pos_oversample'], 1.0).astype(np.float32)
sampler = WeightedRandomSampler(
    weights=sample_weights.tolist(),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    sampler=sampler,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)



Model parameters: 368,195

In [8]:
print(config['loss_weights'], config.get('positive_ltv_weight'))

[1.0, 0.0, 0.0] 4.0


In [9]:
print(len(train_dataset), len(val_dataset), len(hold_dataset))

1895 406 407


In [10]:
print([c for c in rfm_df.columns if c.startswith('actual_ltv_')])

['actual_ltv_12m', 'actual_ltv_24m', 'actual_ltv_36m']


In [11]:
# Set up W&B
wandb_run = None
if USE_WANDB:
    try:
        import wandb
        wandb_run = wandb.init(
            project=settings.WANDB_PROJECT,
            name='week3_transformer',
            tags=['week3', 'transformer', 'pytorch'],
            config=config,
        )
        print(f'W&B run: {wandb_run.name}')
    except Exception as e:
        print(f'W&B init failed: {e}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rakshitkaintura2005 (rakshitkaintura2005-sir-m-visvesvaraya-institute-of-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: week3_transformer


In [12]:
trainer = LTVTrainer(model, config, device=DEVICE, wandb_run=wandb_run)

rprint('[bold blue]Training Transformer...[/bold blue]')
history = trainer.train(
    train_loader, val_loader,
    epochs=TRAIN_EPOCHS,
    patience=PATIENCE,
    checkpoint_dir=MODELS_DIR,
)

rprint(f'\nBest val loss: {trainer.best_val_loss:.6f} at epoch {trainer.best_epoch}')

2026-04-26 01:41:41.827 | INFO     | backend.ml.trainer:__init__:109 - Trainer device: cpu


Training Transformer...

2026-04-26 01:41:42.981 | INFO     | backend.ml.trainer:train:225 - Training 50 epochs | 1450 steps | warmup=145 | device=cpu
2026-04-26 01:41:50.792 | INFO     | backend.ml.trainer:save_checkpoint:312 - Checkpoint saved → E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer_best.pt
2026-04-26 01:41:50.794 | INFO     | backend.ml.trainer:train:270 - Epoch   1/50 | train=4.4946 val=3.8162 mae12m=1724.98 mae36m=1728.58 lr=2.00e-04 [7.8s]
2026-04-26 01:41:59.757 | INFO     | backend.ml.trainer:save_checkpoint:312 - Checkpoint saved → E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer_best.pt
2026-04-26 01:41:59.759 | INFO     | backend.ml.trainer:train:270 - Epoch   2/50 | train=3.6811 val=3.4984 mae12m=1723.03 mae36m=1728.54 lr=4.00e-04 [9.0s]
2026-04-26 01:42:08.312 | INFO     | backend.ml.trainer:save_checkpoint:312 - Checkpoint saved → E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALU

Best val loss: 1.100169 at epoch 50

## 4. Training Curves

In [13]:
fig = make_subplots(rows=1, cols=3, subplot_titles=['Loss', 'MAE 12m', 'Learning Rate'])
epochs_x = list(range(1, len(history['train_loss']) + 1))
fig.add_trace(go.Scatter(x=epochs_x, y=history['train_loss'],   name='Train', line=dict(color='steelblue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history['val_loss'],     name='Val',   line=dict(color='crimson')),  row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history['val_mae_12m'],  name='MAE',   line=dict(color='green')),    row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=history['lr'],           name='LR',    line=dict(color='orange')),   row=1, col=3)
best_epoch = int(trainer.best_epoch)
fig.add_shape(type='line', x0=best_epoch, x1=best_epoch, y0=0, y1=1,
              xref='x1', yref='paper', line=dict(dash='dash', color='gold'))
fig.add_annotation(x=best_epoch, y=1, xref='x1', yref='paper',
                   text=f'Best: ep{best_epoch}', showarrow=False, yshift=10)
fig.update_layout(height=380, title='Transformer Training Curves', showlegend=True)
fig.show()

## 5. Holdout Evaluation

In [14]:
hold_loader = DataLoader(hold_dataset, batch_size=128, shuffle=False,
                         collate_fn=collate_fn, num_workers=0)

rprint('[bold blue]Evaluating baseline transformer on holdout...[/bold blue]')
metrics_raw = evaluate_on_holdout(model, hold_loader, device=DEVICE)

# Two-stage model on top of frozen transformer embeddings with isotonic calibration:
#  1) P(buy) classifier + isotonic calibration
#  2) E(spend | buy) regressor + isotonic calibration
# Final prediction = P(buy)_cal * Spend_cal, then blended with raw transformer prediction
from backend.ml.bgnbd_model import compute_gini, compute_top_decile_lift, compute_calibration_error
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

@torch.no_grad()
def _collect_embeddings_and_targets(loader, model, device):
    model.eval()
    xs, y12, y36 = [], [], []
    for batch in loader:
        tokens = {k: v.to(device) for k, v in batch['tokens'].items()}
        out = model(tokens, return_embedding=True)
        xs.append(out['embedding'].cpu().numpy())
        y12.extend(batch['targets']['ltv_12m'].numpy())
        y36.extend(batch['targets']['ltv_36m'].numpy())
    return np.vstack(xs), np.asarray(y12, dtype=float), np.asarray(y36, dtype=float)

@torch.no_grad()
def _collect_raw_12m_preds(loader, model, device):
    model.eval()
    preds = []
    for batch in loader:
        tokens = {k: v.to(device) for k, v in batch['tokens'].items()}
        out = model(tokens)
        preds.extend(out['ltv_12m'].cpu().numpy())
    return np.asarray(preds, dtype=float)

X_train, y_train_12, _ = _collect_embeddings_and_targets(train_loader, model, DEVICE)
X_val,   y_val_12,   _ = _collect_embeddings_and_targets(val_loader,   model, DEVICE)
X_hold,  y_hold_12, y_hold_36 = _collect_embeddings_and_targets(hold_loader, model, DEVICE)

p_raw_val  = _collect_raw_12m_preds(val_loader, model, DEVICE)
p_raw_hold = _collect_raw_12m_preds(hold_loader, model, DEVICE)

buy_train = (y_train_12 > 0).astype(int)
buy_val   = (y_val_12 > 0).astype(int)

# Stage 1: probability of any purchase
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_hold_s  = scaler.transform(X_hold)

clf = LogisticRegression(
    max_iter=3000,
    class_weight='balanced',
    C=0.5,
    solver='lbfgs',
)
clf.fit(X_train_s, buy_train)

p_buy_val_raw  = clf.predict_proba(X_val_s)[:, 1]
p_buy_hold_raw = clf.predict_proba(X_hold_s)[:, 1]

# Calibrate buy probability with isotonic on validation
iso_buy = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')
iso_buy.fit(p_buy_val_raw, buy_val.astype(float))
p_buy_val  = iso_buy.predict(p_buy_val_raw)
p_buy_hold = iso_buy.predict(p_buy_hold_raw)

# Stage 2: spend conditional on purchase (train on positives only)
pos_mask_train = y_train_12 > 0
if pos_mask_train.sum() < 50:
    raise ValueError('Too few positive LTV rows for stage-2 regression.')

reg = HistGradientBoostingRegressor(
    loss='squared_error',
    learning_rate=0.05,
    max_depth=6,
    max_iter=600,
    min_samples_leaf=20,
    l2_regularization=1e-3,
    random_state=42,
)
reg.fit(X_train_s[pos_mask_train], np.log1p(y_train_12[pos_mask_train]))

# Raw spend predictions
spend_val_raw  = np.clip(np.expm1(reg.predict(X_val_s)), 0, None)
spend_hold_raw = np.clip(np.expm1(reg.predict(X_hold_s)), 0, None)

# Calibrate spend with isotonic on validation positives
pos_mask_val = y_val_12 > 0
if pos_mask_val.sum() >= 20:
    iso_spend = IsotonicRegression(y_min=0.0, out_of_bounds='clip')
    iso_spend.fit(spend_val_raw[pos_mask_val], y_val_12[pos_mask_val])
    spend_val_cal  = iso_spend.predict(spend_val_raw)
    spend_hold_cal = iso_spend.predict(spend_hold_raw)
else:
    spend_val_cal, spend_hold_cal = spend_val_raw, spend_hold_raw

# Tail control to reduce calibration spikes
q99_val = float(np.quantile(spend_val_cal, 0.99)) if len(spend_val_cal) else 0.0
tail_cap = max(1.0, q99_val * 1.2)
spend_val_cal  = np.clip(spend_val_cal,  0, tail_cap)
spend_hold_cal = np.clip(spend_hold_cal, 0, tail_cap)

# Two-stage predictions
pred_val_stage2  = p_buy_val * spend_val_cal
pred_hold_stage2 = p_buy_hold * spend_hold_cal

# Blend with raw transformer prediction; optimize validation objective (MAE + calibration)
lambdas = np.linspace(0.0, 1.0, 21)
val_objs = []
for lam in lambdas:
    pv = (1 - lam) * p_raw_val + lam * pred_val_stage2
    mae = np.mean(np.abs(y_val_12 - pv))
    cal = compute_calibration_error(y_val_12, pv)
    val_objs.append(mae + 0.25 * cal * max(y_val_12.mean(), 1.0))

best_lam = float(lambdas[int(np.argmin(val_objs))])
pred_hold_12 = (1 - best_lam) * p_raw_hold + best_lam * pred_hold_stage2

mean_ltv = float(y_hold_12.mean()) if y_hold_12.mean() > 0 else 1.0
metrics = {
    'mae_ltv_12m':       float(np.mean(np.abs(y_hold_12 - pred_hold_12))),
    'mae_ltv_36m':       metrics_raw['mae_ltv_36m'],
    'mae_pct_12m':       float(np.mean(np.abs(y_hold_12 - pred_hold_12)) / mean_ltv),
    'rmse_ltv_12m':      float(np.sqrt(np.mean((y_hold_12 - pred_hold_12) ** 2))),
    'gini_coefficient':  compute_gini(y_hold_12, pred_hold_12),
    'top_decile_lift':   compute_top_decile_lift(y_hold_12, pred_hold_12),
    'calibration_error': compute_calibration_error(y_hold_12, pred_hold_12),
    'mean_actual_ltv':   mean_ltv,
    'n_customers':       len(y_hold_12),
}

rprint(f"[bold]Two-stage blend:[/bold] lambda={best_lam:.2f} | tail_cap={tail_cap:.2f}")
rprint('[bold]Two-stage calibrated holdout metrics:[/bold]')
rprint(f"  MAE 12m:          GBP {metrics['mae_ltv_12m']:.2f} ({100*metrics['mae_pct_12m']:.1f}% of mean)")
rprint(f"  Gini:             {metrics['gini_coefficient']:.4f}")
rprint(f"  Top decile lift:  {metrics['top_decile_lift']:.2f}x")
rprint(f"  Calibration err:  {metrics['calibration_error']:.4f}")

targets_met = {
    'MAE 12m < 15%':           metrics['mae_pct_12m'] < 0.15,
    'Gini > 0.65':             metrics['gini_coefficient'] > 0.65,
    'Top decile lift > 3.0x':  metrics['top_decile_lift'] > 3.0,
    'Calibration error < 10%': metrics['calibration_error'] < 0.10,
}

rprint('\n[bold]Metric Targets:[/bold]')
for target, passed in targets_met.items():
    status = '[bold green]PASS[/bold green]' if passed else '[bold red]FAIL[/bold red]'
    rprint(f'  {status}  {target}')



Evaluating baseline transformer on holdout...

2026-04-26 01:50:20.332 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:80 - === Transformer Holdout Metrics ===
2026-04-26 01:50:20.333 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:81 -   MAE LTV 12m:        £1370.98  (93.6% of mean)
2026-04-26 01:50:20.333 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:83 -   MAE LTV 36m:        £1465.31
2026-04-26 01:50:20.334 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:84 -   Gini coefficient:   0.5811  (target > 0.65)
2026-04-26 01:50:20.334 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:85 -   Top decile lift:    2.60×  (target > 3.0×)
2026-04-26 01:50:20.335 | INFO     | backend.ml.transformer_evaluator:evaluate_on_holdout:86 -   Calibration error:  0.4544  (target < 0.10)


Two-stage blend: lambda=0.55 | tail_cap=56136.55

Two-stage calibrated holdout metrics:

MAE 12m:          GBP 1243.70 (84.9% of mean)

...

Metric Targets:

FAIL  MAE 12m < 15%

PASS  Gini > 0.65

PASS  Top decile lift > 3.0x

FAIL  Calibration error < 10%

## 6. ONNX Export + Validation

In [15]:
rprint('[bold blue]Exporting to ONNX...[/bold blue]')
onnx_path = export_to_onnx(model, ONNX_PATH, max_seq_len=MAX_SEQ_LEN)
rprint(f'✓ ONNX saved: {onnx_path}')

# Validate parity
rprint('\n[bold blue]Validating ONNX vs PyTorch...[/bold blue]')
val_result = validate_onnx_vs_pytorch(
    model, onnx_path,
    max_seq_len=MAX_SEQ_LEN,
    n_test_samples=200,
    tolerance=1e-4,
)

rprint(f'  MAE diff:    {val_result["mae_diff"]:.2e}  (target: < 1e-5)')
rprint(f'  Max diff:    {val_result["max_diff"]:.2e}')
rprint(f'  PyTorch:     {val_result["latency_pytorch_ms"]:.2f} ms/sample')
rprint(f'  ONNX RT:     {val_result["latency_onnx_ms"]:.2f} ms/sample')
rprint(f'  Speedup:     {val_result["speedup"]:.2f}×')
rprint(f'  Parity OK:   {"✓" if val_result["passed"] else "✗"}')

Exporting to ONNX...

2026-04-26 01:50:25.310 | INFO     | backend.ml.transformer_onnx:export_to_onnx:95 - Exporting model to ONNX: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer.onnx
2026-04-26 01:50:25.770 | INFO     | backend.ml.transformer_onnx:export_to_onnx:133 - ONNX export complete — 1.56 MB saved to E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer.onnx


✓ ONNX saved: E:\DOWNLOADS\DATA SCIENCE\PROFESSIONAL PROJECTS\CUSTOMER_LIFE_TIME_VALUE\models\transformer.onnx

Validating ONNX vs PyTorch...

2026-04-26 01:50:26.218 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:210 - === ONNX Validation ===
2026-04-26 01:50:26.219 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:211 -   Max diff:          1.83e-04  (tolerance: 1.00e-04)
2026-04-26 01:50:26.220 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:212 -   MAE diff:          2.09e-05
2026-04-26 01:50:26.220 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:213 -   PyTorch latency:   0.67 ms/sample
2026-04-26 01:50:26.220 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:214 -   ONNX RT latency:   0.74 ms/sample
2026-04-26 01:50:26.221 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:215 -   Speedup:           0.92×
2026-04-26 01:50:26.221 | INFO     | backend.ml.transformer_onnx:validate_onnx_vs_pytorch:216 -   Parity check:      ✓ PASSED


...

Max diff:    1.83e-04

PyTorch:     0.67 ms/sample

ONNX RT:     0.74 ms/sample

Speedup:     0.92×

Parity OK:   ✓

In [16]:
# Benchmark ONNX latency
engine = ONNXInferenceEngine(onnx_path)
bench  = engine.benchmark(max_seq_len=MAX_SEQ_LEN, n_runs=500)

rprint(f'\n[bold]ONNX Latency Benchmark:[/bold]')
rprint(f'  Mean:   {bench["mean_ms"]:.2f} ms  (target < 200ms)')
rprint(f'  Median: {bench["median_ms"]:.2f} ms')
rprint(f'  P95:    {bench["p95_ms"]:.2f} ms')
rprint(f'  P99:    {bench["p99_ms"]:.2f} ms')
rprint(f'  Target met: {"✓" if bench["p99_ms"] < 200 else "✗"}')

2026-04-26 01:50:26.326 | INFO     | backend.ml.transformer_onnx:__init__:274 - ONNX Runtime session loaded — 4 inputs, 3 outputs
2026-04-26 01:50:26.332 | INFO     | backend.ml.transformer_onnx:warmup:338 - ONNX Runtime warmed up (5 passes)
2026-04-26 01:50:26.746 | INFO     | backend.ml.transformer_onnx:benchmark:365 - ONNX benchmark — mean=0.83ms median=0.80ms p95=1.03ms p99=1.33ms


ONNX Latency Benchmark:

Mean:   0.83 ms  (target < 200ms)

Median: 0.80 ms

P95:    1.03 ms

P99:    1.33 ms

Target met: ✓

## 7. Predict All Customers + Store Embeddings

In [17]:
# Full predictions with MC Dropout uncertainty
full_dataset = PurchaseSequenceDataset(
    sequences_df,
    rfm_df,
    max_length=MAX_SEQ_LEN,
    ltv_12m_col='actual_ltv_12m',
    ltv_24m_col='actual_ltv_24m',
    ltv_36m_col='actual_ltv_36m',
)

rprint('[bold blue]Generating predictions + MC Dropout uncertainty...[/bold blue]')
predictions = predict_all_customers(model, full_dataset, device=DEVICE, n_mc_samples=50)

print(f'Predictions: {len(predictions):,} customers')
print(f'Mean LTV 36m: GBP {predictions["ltv_36m"].mean():.2f}')
display(predictions.head(5))



Generating predictions + MC Dropout uncertainty...

Predictions: 2,708 customers
Mean LTV 36m: GBP 0.20


customer_id,ltv_12m,ltv_24m,ltv_36m,ltv_12m_mean,ltv_12m_std,ltv_36m_mean,ltv_36m_std,ltv_12m_lower,ltv_12m_upper,ltv_36m_lower,ltv_36m_upper
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""14823""",605.34491,0.017068,0.127521,600.425232,54.280609,0.165493,0.170009,510.077545,672.481079,0.021761,0.467133
"""17967""",605.332092,0.017176,0.126765,609.869202,43.737476,0.191204,0.167688,532.192932,672.444641,0.04658,0.486686
"""14498""",605.328674,0.01715,0.126566,606.074829,55.493702,0.153633,0.133133,510.142792,672.480469,0.023504,0.458665
"""16726""",605.358826,0.017147,0.124295,603.027283,62.846203,0.16,0.270252,479.212524,672.532471,0.020104,0.346159
"""14133""",605.328125,0.017261,0.126744,605.112244,60.154995,0.187491,0.181823,493.990692,672.49585,0.051476,0.529953


In [18]:
# Extract and store CLS embeddings
MODEL_VERSION = 'transformer_v1'

rprint('[bold blue]Extracting CLS embeddings for pgvector...[/bold blue]')
customer_ids, embeddings = extract_embeddings(model, full_dataset, device=DEVICE)
print(f'Embeddings shape: {embeddings.shape}')
print(f'Embedding range: [{embeddings.min():.3f}, {embeddings.max():.3f}]')

Extracting CLS embeddings for pgvector...

2026-04-26 01:51:34.424 | INFO     | backend.ml.embedding_store:extract_embeddings:67 - Extracted 2708 embeddings - shape (2708, 96) - dtype float32


Embeddings shape: (2708, 96)
Embedding range: [-5.172, 4.024]


## 8. LTV Distribution Charts

In [19]:
p99_q = predictions['ltv_36m'].quantile(0.99)
if p99_q is None:
    raise ValueError('Cannot compute p99 from empty predictions.')
p99 = float(p99_q)
pred_trimmed = predictions.filter(pl.col('ltv_36m') <= p99)

fig = make_subplots(rows=1, cols=2,
                     subplot_titles=['LTV 36m Distribution', 'MC Dropout Uncertainty (std)'])
fig.add_trace(go.Histogram(x=pred_trimmed['ltv_36m'].to_list(), nbinsx=70,
                            name='LTV 36m', showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=predictions['ltv_36m_std'].to_list(), nbinsx=60,
                            name='Std', showlegend=False), row=1, col=2)
fig.update_layout(height=350, title='Transformer LTV Predictions')
fig.show()

In [20]:
# Uncertainty vs LTV
sample = predictions.filter(pl.col('ltv_36m') <= p99).sample(min(2000, len(predictions)))
fig = px.scatter(
    sample.to_pandas(),
    x='ltv_36m', y='ltv_36m_std',
    opacity=0.4,
    title='LTV 36m Prediction vs MC Dropout Uncertainty',
    labels={'ltv_36m': 'LTV 36m (£)', 'ltv_36m_std': 'Uncertainty (std)'}
)
fig.show()

## 9. Save to Supabase + W&B

In [21]:
if SAVE_TO_DB:
    import uuid
    from datetime import datetime, timezone

    db = SupabaseClient(use_service_role=True)
    assert db.health_check(), 'DB health check failed'

    run_id = str(uuid.uuid4())[:8]

    # Save model registry
    print('Saving model registry...')
    db.bulk_upsert('transformer_model_registry', [{
        'model_version':        MODEL_VERSION,
        'trained_at':           datetime.now(timezone.utc).isoformat(),
        'observation_end':      str(obs_end),
        'embedding_dim':        config['model_dim'],
        'n_layers':             config['n_layers'],
        'n_heads':              config['n_heads'],
        'ffn_dim':              config['ffn_dim'],
        'dropout':              config['dropout'],
        'max_sequence_length':  MAX_SEQ_LEN,
        'learning_rate':        config['learning_rate'],
        'batch_size':           config['batch_size'],
        'epochs_trained':       trainer.best_epoch,
        'optimizer':            'AdamW',
        'loss_function':        'MultiHorizonHuber',
        'train_loss_final':     history['train_loss'][-1],
        'val_loss_final':       history['val_loss'][-1],
        'best_val_loss':        trainer.best_val_loss,
        'best_epoch':           trainer.best_epoch,
        'mae_ltv_12m':          metrics['mae_ltv_12m'],
        'mae_ltv_36m':          metrics['mae_ltv_36m'],
        'mae_pct_12m':          metrics['mae_pct_12m'],
        'gini_coefficient':     metrics['gini_coefficient'],
        'top_decile_lift':      metrics['top_decile_lift'],
        'calibration_error':    metrics['calibration_error'],
        'onnx_exported':        True,
        'onnx_path':            str(ONNX_PATH),
        'onnx_inference_ms_avg':bench['mean_ms'],
        'onnx_pytorch_mae_delta':val_result['mae_diff'],
        'wandb_run_id':         wandb_run.id if wandb_run else None,
        'pipeline_run_id':      run_id,
    }], conflict_columns=['model_version'])

    # Save predictions
    print('Saving transformer predictions...')
    pred_records = predictions.with_columns(
        pl.lit(MODEL_VERSION).alias('model_version'),
        pl.lit(datetime.now(timezone.utc).isoformat()).alias('predicted_at'),
        pl.lit('pytorch').alias('inference_backend'),
    ).to_dicts()
    n_pred = db.bulk_upsert('transformer_predictions', pred_records,
                             conflict_columns=['customer_id', 'model_version'])
    print(f'OK {n_pred:,} predictions saved')

    # Save embeddings to pgvector
    print('Storing CLS embeddings in pgvector...')
    n_emb = store_embeddings(customer_ids, embeddings, MODEL_VERSION, db)
    print(f'OK {n_emb:,} embeddings stored')

    print(f'\nWeek 3 complete. run_id={run_id}')
else:
    print('Skipping DB save')



2026-04-26 01:52:28.377 | INFO     | backend.db.supabase_client:get_supabase_admin_client:47 - Supabase admin client initialised (service-role)
2026-04-26 01:52:28.396 | INFO     | backend.db.supabase_client:get_db_engine:77 - SQLAlchemy sync engine created — pool_size=5, max_overflow=10


Saving model registry...


2026-04-26 01:52:29.865 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 1/1 into transformer_model_registry (1 rows)
2026-04-26 01:52:29.925 | INFO     | backend.db.supabase_client:bulk_upsert:227 - bulk_upsert → 1 rows into transformer_model_registry


Saving transformer predictions...


2026-04-26 01:52:59.905 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 1/6 into transformer_predictions (500 rows)
2026-04-26 01:53:33.260 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 2/6 into transformer_predictions (500 rows)
2026-04-26 01:54:08.098 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 3/6 into transformer_predictions (500 rows)
2026-04-26 01:54:43.417 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 4/6 into transformer_predictions (500 rows)
2026-04-26 01:55:17.777 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 5/6 into transformer_predictions (500 rows)
2026-04-26 01:55:31.341 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 6/6 into transformer_predictions (208 rows)
2026-04-26 01:55:31.400 | INFO     | backend.db.supabase_client:bulk_upsert:227 - bulk_upsert → 2708 rows into transformer_predictions


OK 2,708 predictions saved
Storing CLS embeddings in pgvector...


2026-04-26 01:55:31.611 | WARNING  | backend.ml.embedding_store:store_embeddings:145 - Embedding dim mismatch for customer_embeddings.embedding: model=96 db=64. Applying automatic alignment (truncate/pad).
2026-04-26 01:55:45.783 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 1/14 into customer_embeddings (200 rows)
2026-04-26 01:55:59.541 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 2/14 into customer_embeddings (200 rows)
2026-04-26 01:56:12.860 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 3/14 into customer_embeddings (200 rows)
2026-04-26 01:56:27.055 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 4/14 into customer_embeddings (200 rows)
2026-04-26 01:56:40.820 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 5/14 into customer_embeddings (200 rows)
2026-04-26 01:56:54.415 | DEBUG    | backend.db.supabase_client:bulk_upsert:219 - Upserted batch 6/14 in

OK 2,708 embeddings stored

Week 3 complete. run_id=233d1d5a


In [22]:
if wandb_run:
    import wandb
    wandb.log({
        **{f'holdout_{k}': v for k, v in metrics.items() if isinstance(v, float)},
        'onnx_mae_diff':       val_result['mae_diff'],
        'onnx_latency_ms_p99': bench['p99_ms'],
        'onnx_speedup':        val_result['speedup'],
    })
    wandb.save(str(ONNX_PATH))
    wandb.finish()
    print('✓ W&B run finished')

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Linked 1 file into the W&B run directory (hardlinks); call wandb.save again to sync new files.


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
holdout_calibration_error,▁
holdout_gini_coefficient,▁
holdout_mae_ltv_12m,▁
holdout_mae_ltv_36m,▁
holdout_mae_pct_12m,▁
holdout_mean_actual_ltv,▁
holdout_rmse_ltv_12m,▁
holdout_top_decile_lift,▁
lr,▂▄▅▇██████▇▇▇▇▇▆▆▆▆▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
+8,...


✓ W&B run finished


In [23]:
print('\n=== Week 3 Summary ===')
print(f'  Model version:    {MODEL_VERSION}')
print(f'  Parameters:       {count_parameters(model):,}')
print(f'  Best epoch:       {trainer.best_epoch}')
print(f'  Best val loss:    {trainer.best_val_loss:.6f}')
print(f'  MAE 12m:          £{metrics["mae_ltv_12m"]:.2f}  ({100*metrics["mae_pct_12m"]:.1f}%)')
print(f'  Gini:             {metrics["gini_coefficient"]:.4f}')
print(f'  Top decile lift:  {metrics["top_decile_lift"]:.2f}×')
print(f'  ONNX parity:      {"✓" if val_result["passed"] else "✗"} (MAE diff {val_result["mae_diff"]:.2e})')
print(f'  ONNX p99 latency: {bench["p99_ms"]:.2f} ms  (target < 200ms)')
print(f'\nNext: notebooks/05_causal_ml.ipynb (Phase 4)')


=== Week 3 Summary ===
  Model version:    transformer_v1
  Parameters:       368,195
  Best epoch:       50
  Best val loss:    1.100169
  MAE 12m:          £1243.70  (84.9%)
  Gini:             0.6979
  Top decile lift:  4.92×
  ONNX parity:      ✓ (MAE diff 2.09e-05)
  ONNX p99 latency: 1.33 ms  (target < 200ms)

Next: notebooks/05_causal_ml.ipynb (Phase 4)
